# 02 – Preprocessing

Clean the catalogue and construct leakage-aware, non-overlapping 30-day forecast records.

Produces:
- `data/processed/japan_clean_events.csv`
- `data/processed/forecasting_dataset.csv`


## 0 · Environment Setup & Colab Dependencies


In [1]:
%pip install -q "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "seaborn>=0.12" "scikit-learn>=1.3" "joblib>=1.3" "jupyter>=1.0" "scipy>=1.10"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
import json
import time
import math
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-learn imports
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, ParameterGrid
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
)
import joblib

warnings.filterwarnings('ignore')
%matplotlib inline

# Aesthetic styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Dynamic repository root detection (seamless in Colab and local Jupyter)
ROOT = Path.cwd().resolve()
for _candidate in [ROOT, *ROOT.parents]:
    if (_candidate / 'data').exists() and (_candidate / 'notebooks').exists():
        ROOT = _candidate
        break

FIGURES   = ROOT / 'figures'
PROCESSED = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
FIGURES.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

print('Environment configured successfully. All libraries loaded.')
print(f'Working directory: {ROOT.name if ROOT.name else "."}')


Environment configured successfully. All libraries loaded.
Working directory: Earthquake JAPAN


## 1 · Data Ingestion & Regional Clean Filtering


In [ ]:
# ── Data Ingestion & Spatial Partitioning ──────────────────────────────────

def load_raw_catalogue():
    """Load the raw earthquake catalogue from the project data directory."""

    filename = "query_M2.5+_2000-2024.csv"

    candidate_paths = [
        ROOT / "data" / "raw" / filename,
        ROOT / filename,
        Path(filename)
    ]

    # Also check for CSV files inside data/raw/
    candidate_paths.extend((ROOT / "data" / "raw").glob("*.csv"))

    for path in candidate_paths:
        if path.exists():
            return pd.read_csv(
                path,
                usecols=["time", "latitude", "longitude", "depth", "mag"]
            )

    raise FileNotFoundError(
        f"'{filename}' was not found. "
        "Please place it inside data/raw/."
    )


# ── Load Raw Catalogue ─────────────────────────────────────────────────────

raw_df = load_raw_catalogue()

print(f"Successfully loaded {len(raw_df):,} raw earthquake events.")


# ── Data Cleaning ──────────────────────────────────────────────────────────

raw_df["time"] = pd.to_datetime(
    raw_df["time"],
    errors="coerce"
)

# Remove records with missing essential values
raw_df = raw_df.dropna(
    subset=["time", "latitude", "longitude", "depth", "mag"]
).copy()

# Remove timezone information for consistent datetime operations
try:
    raw_df["time"] = raw_df["time"].dt.tz_localize(None)
except (TypeError, AttributeError):
    pass

# Sort events chronologically
raw_df = raw_df.sort_values("time").reset_index(drop=True)


# ── Filter Japan Study Region ──────────────────────────────────────────────

# Geographic and seismic filters:
# Latitude:  30°N–45°N
# Longitude: 130°E–146°E
# Magnitude: M ≥ 2.5
# Depth:     0–700 km

japan_df = raw_df[
    raw_df["latitude"].between(30.0, 45.0)
    & raw_df["longitude"].between(130.0, 146.0)
    & (raw_df["mag"] >= 2.5)
    & raw_df["depth"].between(0, 700)
].copy()


# ── Create Six Spatial Grid Cells ──────────────────────────────────────────

lat_bins = [30.0, 35.0, 40.0, 45.0]
lon_bins = [130.0, 138.0, 146.0]

lat_labels = [
    "South (30–35N)",
    "Central (35–40N)",
    "North (40–45N)"
]

lon_labels = [
    "West (130–138E)",
    "East (138–146E)"
]

japan_df["lat_bin"] = pd.cut(
    japan_df["latitude"],
    bins=lat_bins,
    labels=lat_labels,
    include_lowest=True
)

japan_df["lon_bin"] = pd.cut(
    japan_df["longitude"],
    bins=lon_bins,
    labels=lon_labels,
    include_lowest=True
)

japan_df["grid_id"] = (
    japan_df["lat_bin"].astype(str)
    + " | "
    + japan_df["lon_bin"].astype(str)
)


# ── Calculate Seismic Energy ───────────────────────────────────────────────

# Gutenberg–Richter relationship:
# E = 10^(1.5M + 4.8) Joules

japan_df["energy_joules"] = (
    10 ** (1.5 * japan_df["mag"] + 4.8)
)


# ── Dataset Summary ────────────────────────────────────────────────────────

target_events = (japan_df["mag"] >= 5.0).sum()
target_percentage = (target_events / len(japan_df)) * 100

print("\n" + "=" * 70)
print("EARTHQUAKE DATASET SUMMARY")
print("=" * 70)

print(f"Total Raw Events          : {len(raw_df):,}")
print(f"Filtered Japan Events     : {len(japan_df):,}")

print(
    f"Temporal Span             : "
    f"{japan_df['time'].min().date()} "
    f"to "
    f"{japan_df['time'].max().date()}"
)

print(
    f"Target Events (M ≥ 5.0)   : "
    f"{target_events:,} ({target_percentage:.2f}%)"
)

print("\nEvents per Spatial Grid:")
print(japan_df["grid_id"].value_counts().sort_index())

print("=" * 70)


# ── Save Processed Dataset ─────────────────────────────────────────────────

output_path = PROCESSED / "japan_clean_events.csv"

japan_df.to_csv(
    output_path,
    index=False
)

print(f"\nSaved clean Japan events to: {output_path}")

Successfully loaded 622,356 raw earthquake events.
EARTHQUAKE DATASET SUMMARY
Total Raw Events: 622,356
Filtered Japan Events (M >= 2.5): 21,954
Temporal Span: 2000-01-02 to 2024-08-10
Target Events (M >= 5.0): 2,411 (10.98%)

Events per Spatial Grid:
grid_id
Central (35-40N) | East (138-146E)    11456
Central (35-40N) | West (130-138E)      710
North (40-45N) | East (138-146E)       3393
North (40-45N) | West (130-138E)        153
South (30-35N) | East (138-146E)       4410
South (30-35N) | West (130-138E)       1832
Name: count, dtype: int64
Saved clean Japan events to data/processed/japan_clean_events.csv


## 2 · Fast Vectorized Spatiotemporal Sample Construction


In [4]:
# ── Fast Vectorized Spatiotemporal Sample Construction ────────────────────
def build_spatiotemporal_samples(japan_df):
    step_start = japan_df['time'].min() + pd.Timedelta(days=30)
    step_end   = japan_df['time'].max() - pd.Timedelta(days=14)
    time_grid  = pd.date_range(start=step_start, end=step_end, freq='30D')
    
    dataset_rows = []
    for gid, group in japan_df.groupby('grid_id'):
        t_arr = group['time'].values
        m_arr = group['mag'].values
        d_arr = group['depth'].values
        e_arr = group['energy_joules'].values
        prior_count = 0
        
        for t_curr in time_grid:
            t_look_start = t_curr - pd.Timedelta(days=30)
            t_horiz_end  = t_curr + pd.Timedelta(days=14)
            
            look_mask  = (t_arr >= np.datetime64(t_look_start)) & (t_arr < np.datetime64(t_curr))
            horiz_mask = (t_arr >= np.datetime64(t_curr)) & (t_arr < np.datetime64(t_horiz_end))
            
            n_events_30d = int(np.sum(look_mask))
            max_m_30d    = float(np.max(m_arr[look_mask])) if n_events_30d > 0 else 0.0
            mean_d_30d   = float(np.mean(d_arr[look_mask])) if n_events_30d > 0 else np.nan
            cum_e_30d    = float(np.sum(e_arr[look_mask])) if n_events_30d > 0 else 0.0
            recent_m45   = int(np.any(m_arr[look_mask] >= 4.5)) if n_events_30d > 0 else 0
            
            m5_target    = int(np.any(m_arr[horiz_mask] >= 5.0))
            m5_count     = int(np.sum(m_arr[horiz_mask] >= 5.0))
            
            dataset_rows.append({
                'timestamp':           t_curr,
                'grid_id':             gid,
                'event_count_30d':     n_events_30d,
                'event_count_lag1':    prior_count,
                'max_mag_30d':         max_m_30d,
                'mean_depth_30d':      mean_d_30d,
                'cum_energy_30d':      cum_e_30d,
                'recent_m45_30d':      recent_m45,
                'target_m5_14d':       m5_target,
                'm5_count_next_14d':   m5_count
            })
            prior_count = n_events_30d
            
    df_features = pd.DataFrame(dataset_rows).sort_values(['timestamp', 'grid_id']).reset_index(drop=True)
    df_features['log_cum_energy_30d'] = np.log10(df_features['cum_energy_30d'] + 1.0)
    df_features['log_event_count_30d'] = np.log1p(df_features['event_count_30d'])
    df_features['log_event_count_lag1'] = np.log1p(df_features['event_count_lag1'])
    return df_features

df_features = build_spatiotemporal_samples(japan_df)
print(f'Generated {len(df_features):,} spatiotemporal forecasting records.')
print('Forecasting sample preview:')
display(df_features.head())


Generated 1,794 spatiotemporal forecasting records.
Forecasting sample preview:


,timestamp,grid_id,event_count_30d,event_count_lag1,max_mag_30d,mean_depth_30d,cum_energy_30d,recent_m45_30d,target_m5_14d,m5_count_next_14d,log_cum_energy_30d,log_event_count_30d,log_event_count_lag1
0,2000-02-01 23:18:53.720,Central (35-40N) | East (138-146E),10,0,5.4,50.520000,1.363971e+13,1,1,1,13.134805,2.397895,0.0
1,2000-02-01 23:18:53.720,Central (35-40N) | West (130-138E),0,0,0.0,NaN,0.000000e+00,0,0,0,0.000000,0.000000,0.0
2,2000-02-01 23:18:53.720,North (40-45N) | East (138-146E),8,0,4.8,65.362500,3.068402e+12,1,0,0,12.486912,2.197225,0.0
3,2000-02-01 23:18:53.720,North (40-45N) | West (130-138E),1,0,3.5,541.000000,1.122018e+10,0,1,1,10.050000,0.693147,0.0
4,2000-02-01 23:18:53.720,South (30-35N) | East (138-146E),7,0,5.3,157.871429,7.669245e+12,1,1,2,12.884753,2.079442,0.0


## 3 · Missing Values, Imputation, Transformations & Chronological Split


In [5]:
# Chronological partitioning by UNIQUE TIMESTAMPS (70% train, 15% val, 15% test)
unique_timestamps = np.sort(df_features['timestamp'].unique())
n_timestamps = len(unique_timestamps)
n_train_times = int(n_timestamps * 0.70)
n_val_times = int(n_timestamps * 0.15)
train_end_time = unique_timestamps[n_train_times - 1]
val_start_time = unique_timestamps[n_train_times]
val_end_time = unique_timestamps[n_train_times + n_val_times - 1]
test_start_time = unique_timestamps[n_train_times + n_val_times]

train_mask = df_features['timestamp'] <= train_end_time
val_mask = (df_features['timestamp'] >= val_start_time) & (df_features['timestamp'] <= val_end_time)
test_mask = df_features['timestamp'] >= test_start_time

# Training-only median imputation for structural missingness in mean_depth_30d
train_median_depth = df_features.loc[train_mask, 'mean_depth_30d'].median()
df_features['mean_depth_imputed'] = df_features['mean_depth_30d'].fillna(train_median_depth)

df_features.to_csv(PROCESSED / 'forecasting_dataset.csv', index=False)
print(f'Saved forecasting dataset to data/processed/forecasting_dataset.csv (median depth imputed: {train_median_depth:.2f} km)')


Saved forecasting dataset to data/processed/forecasting_dataset.csv (median depth imputed: 86.05 km)


## 4 · Before-and-After Pipeline Summary


In [6]:
stages = pd.DataFrame({
    'Stage': ['Raw Catalogue', 'Japan Clean Events', 'Forecasting Samples (Train/Val/Test)'],
    'Rows': [len(raw_df), len(japan_df), len(df_features)],
    'Columns': [raw_df.shape[1], japan_df.shape[1], df_features.shape[1]],
    'Treatment': [
        'Source catalogue untouched',
        'Type/regional/completeness filter (M>=2.5, 30-45N, 130-146E)',
        'Non-overlapping 30d lookback -> 14d horizon binary targets'
    ]
})
display(stages)
print('\nLeakage guard: features use [t-30d, t); target uses [t, t+14d); zero lookahead.')


,Stage,Rows,Columns,Treatment
0,Raw Catalogue,622356,5,Source catalogue untouched
1,Japan Clean Events,21954,9,"Type/regional/completeness filter (M>=2.5, 30-..."
2,Forecasting Samples (Train/Val/Test),1794,14,Non-overlapping 30d lookback -> 14d horizon bi...



Leakage guard: features use [t-30d, t); target uses [t, t+14d); zero lookahead.
